# RL Exploration Lab -- Interactive Walkthrough

This notebook provides a hands-on tour of **RL Exploration Lab**, a library of **17 reinforcement learning exploration methods** evaluated under a unified protocol on MiniGrid environments.

Methods range from simple baselines (epsilon-greedy, count-based) through prediction-error approaches (RND, ICM) to language-grounded exploration (SHELM, L-AMIGo). Every method inherits from a single `BaseExploration` interface, making them drop-in replaceable with one config change.

**What this notebook covers:**
1. Environment setup and observation space
2. Instantiating and comparing exploration methods
3. Running a quick training experiment
4. Architecture overview

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Environment Overview

MiniGrid provides procedurally generated grid-world environments with partial observability. The agent sees a 7x7x3 local view (flattened to 147 dims by our wrapper). We use **KeyCorridorS3R2** as the main benchmark -- the agent must navigate corridors, pick up a key, and unlock a door.

In [ ]:
from rl_exploration_lab.envs.minigrid_wrapper import make_wrapped_env

env = make_wrapped_env("KeyCorridorS3R2", seed=SEED, render_mode="rgb_array")

obs, info = env.reset()

print(f"Environment:       MiniGrid-KeyCorridorS3R2")
print(f"Observation shape: {obs.shape}  (7x7x3 grid view, flattened & normalized)")
print(f"Action space:      Discrete({env.n_actions})  [left, right, forward, pickup, drop, toggle, done]")
print(f"Obs dtype:         {obs.dtype}")
print(f"Obs range:         [{obs.min():.2f}, {obs.max():.2f}]")

In [ ]:
# Render the environment
frame = env.render()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full rendered view
axes[0].imshow(frame)
axes[0].set_title("KeyCorridorS3R2 (rendered view)")
axes[0].axis("off")

# Agent's partial observation (7x7x3)
obs_image = obs.reshape(7, 7, 3)
axes[1].imshow(obs_image, interpolation="nearest")
axes[1].set_title("Agent's 7x7x3 partial observation")
axes[1].axis("off")

plt.tight_layout()
plt.show()

env.close()

## 3. Exploration Method Comparison

Every exploration method inherits from `BaseExploration` and implements `compute_intrinsic_reward(obs, next_obs, action)`. This uniform interface makes methods interchangeable -- swap them with a single config flag.

We compare three representative methods:
- **none** (Epsilon-Greedy baseline) -- no intrinsic reward
- **count_based** (Bellemare et al., 2016) -- reward inversely proportional to visit count
- **rnd** (Burda et al., 2018) -- reward from prediction error of a random target network

In [ ]:
from rl_exploration_lab.evaluation.evaluator import create_exploration

OBS_DIM = 147  # 7 * 7 * 3

methods = {
    "none (baseline)": create_exploration("none", obs_dim=OBS_DIM, device=device),
    "count_based":     create_exploration("count_based", obs_dim=OBS_DIM, device=device),
    "rnd":             create_exploration("rnd", obs_dim=OBS_DIM, device=device),
}

for name, method in methods.items():
    print(f"{name:20s} -> {method.__class__.__name__}")

In [ ]:
# Generate dummy transitions and compute intrinsic rewards
batch_size = 64
dummy_obs = torch.randn(batch_size, OBS_DIM, device=device)
dummy_next_obs = torch.randn(batch_size, OBS_DIM, device=device)
dummy_actions = torch.randint(0, 7, (batch_size,), device=device)

reward_means = {}
reward_stds = {}

for name, method in methods.items():
    r_int = method.compute_intrinsic_reward(dummy_obs, dummy_next_obs, dummy_actions)
    reward_means[name] = r_int.mean().item()
    reward_stds[name] = r_int.std().item()
    print(f"{name:20s} | mean intrinsic reward: {reward_means[name]:.4f} +/- {reward_stds[name]:.4f}")

In [ ]:
# Visualize intrinsic reward magnitudes
names = list(reward_means.keys())
means = [reward_means[n] for n in names]
stds = [reward_stds[n] for n in names]

colors = ["#6c757d", "#0d6efd", "#dc3545"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Mean Intrinsic Reward")
ax.set_title("Intrinsic Reward Magnitude by Exploration Method")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{m:.4f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 4. Quick Training Run

We run PPO + RND on the simple **Empty-8x8** environment for 50K steps to demonstrate the training pipeline. This should complete in under a minute on CPU.

> **Note:** For full benchmarks across all 17 methods and 5 environments, see `scripts/evaluate.py --suite full`.

In [ ]:
from rl_exploration_lab.evaluation.evaluator import run_single_experiment

config = {
    "total_steps": 50_000,
    "rollout_steps": 2048,
    "batch_size": 64,
    "n_epochs": 4,
    "lr": 2.5e-4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.2,
    "value_coef": 0.5,
    "entropy_coef": 0.01,
    "max_grad_norm": 0.5,
    "intrinsic_coef": 0.01,
    "hidden_dim": 128,
    "embed_dim": 64,
    "exploration_kwargs": {},
}

print("Training PPO + RND on Empty-8x8 (50K steps)...")
print(f"Device: {device}\n")

result = run_single_experiment(
    method_name="rnd",
    env_name="Empty-8x8",
    seed=0,
    config=config,
    device=device,
)

print(f"\n{'='*50}")
print(f"  Mean reward:  {result.mean_reward:.4f}")
print(f"  Solve rate:   {result.solve_rate:.1%}")
print(f"  Episodes:     {len(result.episode_rewards)}")
print(f"  Time:         {result.elapsed_seconds:.1f}s")
print(f"{'='*50}")
print("\nFor full benchmarks, see: python scripts/evaluate.py --suite full")

## 5. Architecture Diagram

All 17 exploration methods share the same interface and plug into the PPO training loop at two points: intrinsic reward computation during rollout collection, and internal updates after each PPO step.

```
BaseExploration (ABC)
  |-- compute_intrinsic_reward(obs, next_obs, action) -> reward
  |-- update(batch) -> metrics
  |
  +-- EpsilonGreedy (baseline)       +-- CountBased
  +-- UCB                            +-- RND
  +-- ICM                            +-- RIDE
  +-- NovelD                         +-- NGU
  +-- AMIGo                          +-- GoExplore
  +-- CLIPRND                        +-- CLIPNovelD
  +-- SemanticExploration            +-- LAMIGo
  +-- LNovelD                        +-- SHELMRND
  +-- SHELMOracle
```

### PPO Training Loop Integration

```
  Environment
      |
      v
  MiniGridWrapper          obs (147,) flat tensor
      |                        |
      v                        v
  Rollout Collection      ActorCritic (policy)
      |                        |
      |--- r_extrinsic         |--- action
      |                        |
      +--- Exploration.compute_intrinsic_reward()
      |         |
      |         +--- r_intrinsic
      |                        |
      v                        v
  r_total = r_ext + beta * r_int
      |
      v
  PPO Update (actor + critic)
      |
      +--- Exploration.update(batch)
      |
      v
  Next rollout ...
```

Config files in `configs/` control all hyperparameters. Switch exploration method with a single flag:
```bash
python scripts/train.py --method rnd    # Random Network Distillation
python scripts/train.py --method icm    # Intrinsic Curiosity Module
python scripts/train.py --method none   # PPO baseline (no intrinsic reward)
```

## 6. Next Steps

**Run the full benchmark** across all methods and environments:
```bash
python scripts/evaluate.py --suite full
```

**Try individual methods:**
```bash
# Go-Explore (archive-based, no GPU needed)
python scripts/train.py --method go_explore --env DoorKey-6x6 --steps 100000

# SHELM + Oracle (thesis proposed improvement)
python scripts/train.py --method shelm_oracle --env KeyCorridorS3R2 --steps 500000

# Generate comparison plots
python scripts/plot_results.py --results results/ --output plots/
```

**Key files:**
- `configs/default.yaml` -- default training hyperparameters
- `rl_exploration_lab/exploration/base.py` -- the `BaseExploration` ABC all methods implement
- `rl_exploration_lab/evaluation/evaluator.py` -- experiment runner and method registry
- `README.md` -- full method list, benchmark environments, and project background